# Benchmark v2: compresión de MAXIM con expertos por tarea

Este notebook mide **costo**, no calidad. No entrena, no evalúa PSNR/SSIM y no
escribe sobre checkpoints. Corrige el protocolo del primer diagnóstico y agrega
las variantes de compresión con pesos nuevos.

**Qué cambia respecto del benchmark v1**

- Todas las variantes se compilan y se cronometran en **un mismo proceso**,
  alternando el orden en cada bloque. En v1 baseline y shared se midieron en
  sesiones distintas, con algoritmos cuDNN distintos para el mismo backbone.
- Caché persistente de compilación en Drive, que también conserva el
  autotuning de XLA entre sesiones cuando el runtime lo permite.
- FLOPs del programa **previo a la optimización**, homogéneos entre variantes,
  además de los FLOPs y bytes del ejecutable compilado, cantidad de kernels
  estimada y algoritmos cuDNN elegidos.
- Shared con **selección efectiva**: se ejecuta sólo la cabeza del experto de
  la tarea.

**Variantes**

| Nombre | Qué es | Pesos |
|---|---|---|
| `B0` | Baseline MAXIM S-2 | checkpoint baseline +10 |
| `S0` | Shared actual, calcula las cinco cabezas | checkpoint shared +10 |
| `S0_sel` | Shared con selección efectiva de la cabeza | checkpoint shared +10 |
| `S1_d32`, `S1_d64` | `S0_sel` con las 8 parejas RCAB del nivel profundo reemplazadas por un módulo común | shared + módulos nuevos aleatorios |
| `S2_d32`, `S2_d64` | Igual, con un banco de 5 módulos por tarea y selección efectiva | shared + módulos nuevos aleatorios |
| `S2_d32_todos` | `S2_d32` calculando los cinco módulos y enmascarando | referencia de selección |
| `S0_sel_sin_rcab2` | `S0_sel` sin esas parejas RCAB | techo de ahorro |
| `ref_sin_cross_gating`, `ref_sin_gmlp` | MAXIM S-2 sin cross-gating o sin gMLP multi-eje | aleatorios, techo de costo |
| `B0_remat` | Baseline con rematerialización por bloque | checkpoint baseline |
| `...@bf16` | Misma variante en bfloat16 | pesos convertidos |

El módulo nuevo es `1×1 C→d → GELU → depthwise 3×3 → 1×1 d→C` con C=128.
Los pesos nuevos sirven para medir costo y shapes, **nunca calidad**.

**Duración estimada en L4:** ~30 min de inferencia y ~60 min de entrenamiento
la primera vez. Cada grupo se guarda al terminar; si el runtime se corta, se
repite sólo el grupo incompleto y la caché acorta las compilaciones.

**Requisito:** el repositorio de GitHub debe contener los módulos
`maxim/models/compression.py` y `maxim/models/compat.py` y la versión de
`maxim/models/maxim.py` que los usa. La celda 1 verifica sus hashes.

In [ ]:
# 1. Entorno: fork con los módulos de compresión
import hashlib
import os
import subprocess
import sys
from pathlib import Path

# MAXIM_BENCH_SMOKE=1 sólo se usa para validar el notebook localmente en CPU
# con un modelo reducido y checkpoints sintéticos. En Colab queda en False.
SMOKE_TEST = os.environ.get("MAXIM_BENCH_SMOKE") == "1"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

REPO_URL = "https://github.com/Matiata/maxim.git"
# Rama o commit que contiene los módulos de compresión.
REPO_REF = "main"
EXPECTED_SOURCE_SHA256 = {
    "maxim/models/maxim.py": "5f8a4e005ffc5b12b5dcfbd9b2a06cab0125dbbdb97813c4b1f8ea1b2d5bf57a",
    "maxim/models/compression.py": "b460019514b05dd0b0bbe828f2d145aa12dbc443e36554b0fc735dbc84636c4c",
    "maxim/models/compat.py": "cebfaa8bcd5eaf947c3d98962a3e2266e44265b16f0836b2d5c878581abe610c"
}


def git(*args):
    return subprocess.check_output(
        ["git", "-C", str(REPO_DIR), *args], text=True
    ).strip()


if SMOKE_TEST:
    REPO_DIR = Path(os.environ["MAXIM_REPO_DIR"]).resolve()
else:
    REPO_DIR = Path("/content/maxim_compresion_src")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    elif git("remote", "get-url", "origin").rstrip("/") != REPO_URL:
        raise RuntimeError("REPO_DIR apunta a otro repositorio.")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    try:
        target = git("rev-parse", "--verify", f"origin/{REPO_REF}")
    except subprocess.CalledProcessError:
        target = REPO_REF
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "checkout", "--detach", target],
        check=True,
    )
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "jax[cuda12]==0.10.2", "flax==0.11.2", "optax==0.2.8",
            "ml-collections==0.1.1", "einops", "Pillow",
        ],
        check=True,
    )

try:
    REPO_COMMIT = git("rev-parse", "HEAD")
    REPO_DIRTY = bool(git("status", "--porcelain", "--", "maxim/models"))
except Exception as error:
    REPO_COMMIT, REPO_DIRTY = f"unavailable: {error}", None

SOURCE_SHA256 = {}
for relative, expected in EXPECTED_SOURCE_SHA256.items():
    path = REPO_DIR / relative
    if not path.exists():
        raise FileNotFoundError(
            f"Falta {relative} en {REPO_REF}. Subir los cambios del repositorio "
            "(git push) o corregir REPO_REF."
        )
    SOURCE_SHA256[relative] = hashlib.sha256(path.read_bytes()).hexdigest()
    if SOURCE_SHA256[relative] != expected:
        raise RuntimeError(
            f"{relative} no coincide con la versión revisada para este "
            f"notebook: {SOURCE_SHA256[relative]} != {expected}."
        )

sys.path.insert(0, str(REPO_DIR))
print("Commit:", REPO_COMMIT, "| modo smoke:", SMOKE_TEST)
print(
    "Si JAX ya estaba importado antes de instalar, reiniciar el runtime y "
    "ejecutar desde el inicio."
)

In [ ]:
# 2. Drive, dependencias y GPU
if not SMOKE_TEST:
    from google.colab import drive

    drive.mount("/content/gdrive")

import collections
import gc
import gzip
import importlib.metadata
import json
import platform
import re
import time
import traceback
from collections.abc import Mapping
from typing import Any

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import traverse_util
from flax.core import unfreeze
from flax.training import checkpoints, train_state

from maxim.models import compression
from maxim.models.compression import (
    VARIANTS,
    build_variant,
    flatten_params,
    is_new_local_mixer,
    parameter_families,
    removed_rcab_conv_at,
    transplant_params,
    variant_replaced_levels,
)

expected_versions = {"jax": "0.10.2", "flax": "0.11.2"}
actual_versions = {
    name: importlib.metadata.version(name) for name in expected_versions
}
if actual_versions != expected_versions:
    raise RuntimeError(
        f"Versiones inesperadas: {actual_versions}; reiniciar el runtime."
    )

DEVICES = jax.devices()
GPU_DEVICES = [device for device in DEVICES if device.platform == "gpu"]
if not GPU_DEVICES and not SMOKE_TEST:
    raise RuntimeError("Seleccionar un runtime GPU en Colab, idealmente L4.")
DEVICE = GPU_DEVICES[0] if GPU_DEVICES else DEVICES[0]
print("Dispositivo:", DEVICE, getattr(DEVICE, "device_kind", ""))

In [ ]:
# 3. Configuración visible antes de medir
TASKS = ["deblur", "dehaze", "denoise", "derain", "enhance"]

if SMOKE_TEST:
    CHECKPOINT_ROOT = Path(os.environ["MAXIM_SMOKE_ROOT"])
else:
    CHECKPOINT_ROOT = Path("/content/gdrive/MyDrive/Facultad/tesis/ckpts")
OUTPUT_DIR = CHECKPOINT_ROOT / "benchmark_maxim_v2_compresion"
RESULT_PATH = OUTPUT_DIR / "benchmark_v2_results.json"
REPORT_PATH = OUTPUT_DIR / "diagnostico_v2_compresion.md"
HLO_DIR = OUTPUT_DIR / "hlo"
COMPILATION_CACHE_DIR = CHECKPOINT_ROOT / "jax_cache_benchmark_maxim_v2"

MODEL_CONFIG = {
    "variant": "S-2",
    "dropout_rate": 0.1,
    "num_outputs": 3,
    "use_bias": True,
    "num_supervision_scales": 3,
}
MODEL_SPECS = {
    "baseline": {
        "relative_run": "maxim_no_moe/baseline_S-2_from_newsplit_ckpt_extra10",
        "epoch": 10,
        "expected_step": 20000,
        "val_macro": 31.874760357289738,
        "template_variant": "B0",
    },
    "shared": {
        "relative_run": "moe_shared_residual_from_newsplit_baseline_extra10",
        "epoch": 8,
        "expected_step": 16000,
        "val_macro": 32.06335479306456,
        "template_variant": "S0",
    },
}

INFERENCE_SHAPES = [(256, 256), (512, 512)]
INFERENCE_VARIANTS = [
    "B0", "S0", "S0_sel",
    "S1_d32", "S2_d32", "S2_d32_todos", "S1_d64", "S2_d64",
    "S0_sel_sin_rcab2", "ref_sin_cross_gating", "ref_sin_gmlp",
    "B0@bf16", "S0_sel@bf16", "S2_d32@bf16",
]
# (variante, alcance entrenable). "new_only" entrena sólo los módulos nuevos
# con el resto congelado, como la primera fase de adaptación prevista.
TRAINING_RUNS = [
    ("B0", "full"),
    ("B0_remat", "full"),
    ("S0", "full"),
    ("S0_sel", "full"),
    ("S1_d32", "new_only"),
    ("S2_d32", "new_only"),
    ("S2_d32", "full"),
]
TRAIN_SHAPE = (256, 256)
TRAIN_BATCH_SIZE = 2
TRAIN_TASK_IDS = [0, 1]  # batch mixto, como el muestreo uniforme por tarea
TRAIN_LEARNING_RATE = 1e-5
TRAIN_WEIGHT_DECAY = 1e-4
AUXILIARY_LOSS_WEIGHT = 1.0
SEED = 42
NEW_MODULE_SEED = 1234
INIT_SHAPE = (64, 64)  # las shapes de los pesos no dependen de la imagen

WARMUP_EXECUTIONS = 10
TIMED_EXECUTIONS = 100
TIMING_BLOCKS = 3
EQUIVALENCE_TOLERANCE = 1e-4

RUN_INFERENCE = True
RUN_TRAINING = True
SAVE_HLO = True
FORCE_REMEASURE = False
STRICT_CHECKS = True  # detener si falla una verificación estructural

if SMOKE_TEST:
    MODEL_CONFIG.update(
        features=8, block_size_hr=(4, 4), grid_size_hr=(4, 4),
        block_size_lr=(2, 2), grid_size_lr=(2, 2),
    )
    INFERENCE_SHAPES = [(32, 32)]
    TRAIN_SHAPE = (32, 32)
    INIT_SHAPE = (32, 32)
    WARMUP_EXECUTIONS, TIMED_EXECUTIONS, TIMING_BLOCKS = 1, 3, 2
    SAVE_HLO = False

for name in INFERENCE_VARIANTS + [run[0] for run in TRAINING_RUNS]:
    if name.split("@")[0] not in VARIANTS:
        raise ValueError(f"Variante desconocida: {name}")
assert len(TRAIN_TASK_IDS) == TRAIN_BATCH_SIZE
assert SMOKE_TEST or (WARMUP_EXECUTIONS >= 10 and TIMED_EXECUTIONS >= 100
                      and TIMING_BLOCKS >= 3)

# Caché persistente: reutiliza ejecutables y, si el runtime lo admite, los
# resultados de autotuning entre sesiones.
CACHE_SETTINGS = {}
for option, value in (
    ("jax_compilation_cache_dir", str(COMPILATION_CACHE_DIR)),
    ("jax_persistent_cache_min_compile_time_secs", 0),
    ("jax_persistent_cache_min_entry_size_bytes", -1),
    ("jax_persistent_cache_enable_xla_caches", "all"),
):
    try:
        jax.config.update(option, value)
        CACHE_SETTINGS[option] = {"value": value, "applied": True}
    except Exception as error:
        CACHE_SETTINGS[option] = {
            "value": value, "applied": False,
            "error": f"{type(error).__name__}: {error}",
        }
COMPILATION_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Resultados:", RESULT_PATH)
print(json.dumps(CACHE_SETTINGS, indent=1))

In [ ]:
# 4. Utilidades: JSON atómico, sincronización, entorno y estadísticas de HLO
def json_safe(value):
    if isinstance(value, Mapping):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not np.isfinite(value):
        return str(value)
    if isinstance(value, (float, int, str, bool)) or value is None:
        return value
    return str(value)


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(json_safe(payload), ensure_ascii=False, indent=2,
                   allow_nan=False),
        encoding="utf-8",
    )
    os.replace(temporary, path)


def block_tree(tree):
    return jax.tree_util.tree_map(
        lambda value: value.block_until_ready()
        if hasattr(value, "block_until_ready") else value,
        tree,
    )


def command_output(command):
    try:
        return subprocess.check_output(
            command, text=True, stderr=subprocess.STDOUT
        ).strip()
    except Exception as error:
        return f"unavailable: {type(error).__name__}: {error}"


def device_memory_stats():
    try:
        return json_safe(DEVICE.memory_stats() or {})
    except Exception as error:
        return {"available": False, "error": f"{type(error).__name__}: {error}"}


def environment_payload():
    packages = {}
    for name in ("jax", "jaxlib", "flax", "optax", "orbax-checkpoint", "numpy"):
        try:
            packages[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            packages[name] = None
    return {
        "python": platform.python_version(),
        "packages": packages,
        "device": str(DEVICE),
        "device_kind": getattr(DEVICE, "device_kind", None),
        "platform": DEVICE.platform,
        "nvidia_smi": command_output([
            "nvidia-smi", "--query-gpu=name,driver_version,memory.total",
            "--format=csv,noheader",
        ]),
        "repo_url": REPO_URL,
        "repo_ref": REPO_REF,
        "repo_commit": REPO_COMMIT,
        "repo_models_dirty": REPO_DIRTY,
        "source_sha256": SOURCE_SHA256,
        "xla_flags": os.environ.get("XLA_FLAGS"),
        "preallocate": os.environ.get("XLA_PYTHON_CLIENT_PREALLOCATE"),
        "default_matmul_precision": str(
            getattr(jax.config, "jax_default_matmul_precision", None)
        ),
        "compilation_cache": CACHE_SETTINGS,
        "smoke_test": SMOKE_TEST,
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }


def entry_computation(text):
    lines = text.splitlines()
    starts = [index for index, line in enumerate(lines)
              if line.startswith("ENTRY")]
    if not starts:
        return []
    body = []
    for line in lines[starts[0] + 1:]:
        if line.startswith("}"):
            break
        body.append(line)
    return body


def hlo_statistics(text):
    """Conteos del ejecutable compilado. Los kernels son una estimación."""
    entry = entry_computation(text)
    fusions = sum(1 for line in entry if re.search(r"\bfusion\(", line))
    custom = [line for line in entry if "custom-call(" in line]
    targets = collections.Counter(
        match.group(1)
        for line in custom
        for match in [re.search(r'custom_call_target="([^"]*)"', line)]
        if match
    )
    algorithms = collections.Counter(
        re.findall(r'"algo_id":"(\d+)"', text)
    )
    conv_ops = text.count(" convolution(")
    return {
        "entry_fusions": fusions,
        "entry_custom_calls": len(custom),
        "kernel_launch_estimate": fusions + len(custom),
        "custom_call_targets": dict(targets),
        "conv_custom_calls": sum(
            count for target, count in targets.items() if "conv" in target
        ),
        "convolution_ops": conv_ops,
        "cudnn_algorithms": dict(algorithms.most_common()),
    }


def stable_key(*parts):
    return "/".join(str(part) for part in parts)

In [ ]:
# 5. Protocolo y resultado reanudable
PROTOCOL = {
    "schema_version": 2,
    "repo_url": REPO_URL,
    "source_sha256": EXPECTED_SOURCE_SHA256,
    "model_config": MODEL_CONFIG,
    "model_specs": MODEL_SPECS,
    "tasks": TASKS,
    "inference_shapes": INFERENCE_SHAPES,
    "inference_variants": INFERENCE_VARIANTS,
    "training_runs": TRAINING_RUNS,
    "train_shape": TRAIN_SHAPE,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "train_task_ids": TRAIN_TASK_IDS,
    "train_learning_rate": TRAIN_LEARNING_RATE,
    "train_weight_decay": TRAIN_WEIGHT_DECAY,
    "seed": SEED,
    "new_module_seed": NEW_MODULE_SEED,
    "warmup_executions": WARMUP_EXECUTIONS,
    "timed_executions": TIMED_EXECUTIONS,
    "timing_blocks": TIMING_BLOCKS,
    "timing": (
        "un proceso; todas las variantes de un grupo compiladas antes de "
        "medir; bloques alternados con orden rotado; tareas ciclando dentro "
        "de cada bloque para variantes con tarea"
    ),
    "inference_output": "sólo la salida final; entradas sintéticas en GPU",
    "training_step": (
        "forward + L1 de supervisión profunda normalizada + backward + "
        "AdamW; estado descartable"
    ),
    "smoke_test": SMOKE_TEST,
}
PROTOCOL_SHA256 = hashlib.sha256(
    json.dumps(json_safe(PROTOCOL), sort_keys=True).encode("utf-8")
).hexdigest()
CURRENT_ENVIRONMENT = environment_payload()


def comparable_environment(payload):
    return {
        "device_kind": payload.get("device_kind"),
        "nvidia_smi": payload.get("nvidia_smi"),
        "packages": {
            name: payload.get("packages", {}).get(name)
            for name in ("jax", "jaxlib", "flax", "optax")
        },
        "source_sha256": payload.get("source_sha256"),
    }


if RESULT_PATH.exists():
    REPORT = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
    if REPORT.get("protocol_sha256") != PROTOCOL_SHA256:
        raise RuntimeError(
            "Existe un resultado de otro protocolo en OUTPUT_DIR. Conservarlo "
            "y cambiar OUTPUT_DIR."
        )
    if comparable_environment(REPORT["environment"]) != comparable_environment(
        CURRENT_ENVIRONMENT
    ):
        raise RuntimeError(
            "La corrida reanudada usa otra GPU, driver, paquetes o código. "
            "Usar un runtime comparable o cambiar OUTPUT_DIR."
        )
else:
    REPORT = {
        "status": "in_progress",
        "protocol_sha256": PROTOCOL_SHA256,
        "protocol": PROTOCOL,
        "environment": CURRENT_ENVIRONMENT,
        "checkpoints": {},
        "variants": {},
        "groups": {},
        "measurements": {},
        "checks": {},
        "limitations": [
            "Entradas sintéticas: miden costo y tiempo, no calidad.",
            "Los módulos nuevos de S1/S2 tienen pesos aleatorios: sólo costo.",
            (
                "La memoria reportada es la estática del ejecutable "
                "(memory_analysis). No es un pico medido de buffers vivos."
            ),
            (
                "Los FLOPs 'lowered' cuentan el programa antes de optimizar y "
                "son comparables entre variantes; los 'compiled' dependen de "
                "las elecciones del backend."
            ),
            (
                "La cantidad de kernels se estima con fusiones y custom calls "
                "del programa compilado; no cuenta kernels internos de cuDNN."
            ),
        ],
    }
REPORT.setdefault("executions", [])
for previous in REPORT["executions"]:
    if previous.get("status") == "started":
        previous["status"] = "interrupted"
REPORT["executions"].append({
    "execution_id": time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()),
    "environment": CURRENT_ENVIRONMENT,
    "status": "started",
})


def persist_report():
    REPORT["updated_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    atomic_json(RESULT_PATH, REPORT)


def group_complete(group):
    return (not FORCE_REMEASURE
            and REPORT["groups"].get(group, {}).get("status") == "complete")


def record_check(group, name, passed, detail):
    REPORT["checks"].setdefault(group, {})[name] = {
        "passed": bool(passed), "detail": json_safe(detail),
    }
    print(("OK   " if passed else "FALLA"), group, name, detail)
    return passed


persist_report()
print("Protocolo:", PROTOCOL_SHA256, "| estado:", REPORT["status"])

In [ ]:
# 6. Restauración estricta y construcción de las variantes
def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def variant_base(name):
    return name.split("@")[0]


def variant_dtype(name):
    return jnp.bfloat16 if name.endswith("@bf16") else jnp.float32


def variant_family(name):
    return VARIANTS[variant_base(name)]["family"]


MODELS = {}


def get_model(name):
    base = variant_base(name)
    if base not in MODELS:
        MODELS[base] = build_variant(base, MODEL_CONFIG, num_tasks=len(TASKS))
    return MODELS[base]


def model_kwargs(name, task_id):
    return {} if variant_family(name) == "baseline" else {"task_id": task_id}


def _init_inputs(name):
    dummy = jnp.zeros((1, *INIT_SHAPE, 3), dtype=jnp.float32)
    return dummy, model_kwargs(name, jnp.zeros((1,), dtype=jnp.int32))


def template_variables(name):
    model = get_model(name)
    dummy, kwargs = _init_inputs(name)
    return jax.eval_shape(
        lambda: model.init(jax.random.PRNGKey(SEED), dummy, train=False,
                           **kwargs)
    )


def initialized_variables(name, seed):
    model = get_model(name)
    dummy, kwargs = _init_inputs(name)
    return jax.jit(
        lambda key: model.init(key, dummy, train=False, **kwargs)
    )(jax.random.PRNGKey(seed))


def validate_tree(expected, actual, label):
    wanted, found = flatten_params(expected), flatten_params(actual)
    missing = sorted(set(wanted) - set(found))
    extra = sorted(set(found) - set(wanted))
    if missing or extra:
        raise ValueError(f"{label}: faltan {missing[:5]}, sobran {extra[:5]}")
    for key in wanted:
        if tuple(wanted[key].shape) != tuple(found[key].shape):
            raise ValueError(f"{label}: shape incompatible en {key}")
        if np.dtype(wanted[key].dtype) != np.dtype(found[key].dtype):
            raise ValueError(f"{label}: dtype incompatible en {key}")


def hash_variables(variables):
    digest = hashlib.sha256()
    for key, value in sorted(flatten_params(variables).items()):
        array = np.ascontiguousarray(np.asarray(value))
        if not np.isfinite(array).all():
            raise FloatingPointError(f"Variable no finita: {key}")
        digest.update(json.dumps([str(part) for part in key]).encode("utf-8"))
        digest.update(str(array.dtype).encode("ascii"))
        digest.update(str(array.shape).encode("ascii"))
        digest.update(array.tobytes())
    return digest.hexdigest()


CHECKPOINTS = {}


def load_checkpoint_strict(kind):
    spec = MODEL_SPECS[kind]
    run_dir = CHECKPOINT_ROOT / spec["relative_run"]
    best_dir = run_dir / "best_checkpoint"
    checkpoint_path = best_dir / f"checkpoint_{spec['epoch']}"
    selection_path = best_dir / "best_metric.json"
    run_metadata_path = run_dir / "run_metadata.json"
    for required in (checkpoint_path, selection_path, run_metadata_path):
        if not required.exists():
            raise FileNotFoundError(required)
    selection = read_json(selection_path)
    if selection.get("selection_metric") != "psnr_macro":
        raise ValueError(f"{kind}: selección no macro.")
    if int(selection.get("epoch", -1)) != spec["epoch"]:
        raise ValueError(f"{kind}: cambió la época seleccionada.")
    if abs(float(selection["psnr_macro"]) - spec["val_macro"]) > 1e-6:
        raise ValueError(f"{kind}: cambió el PSNR macro seleccionado.")
    if selection.get("task_names") != TASKS:
        raise ValueError(f"{kind}: orden de tareas incompatible.")
    run_metadata = read_json(run_metadata_path)
    if "split_manifest" not in run_metadata:
        raise ValueError(f"{kind}: falta split_manifest.")
    split_hash = hashlib.sha256(
        json.dumps(run_metadata["split_manifest"], sort_keys=True)
        .encode("utf-8")
    ).hexdigest()

    raw = checkpoints.restore_checkpoint(
        ckpt_dir=str(checkpoint_path), target=None
    )
    if not isinstance(raw, Mapping) or "params" not in raw or "step" not in raw:
        raise ValueError(f"{kind}: no se restauró un TrainState completo.")
    step = int(np.asarray(raw["step"]))
    if step != spec["expected_step"]:
        raise ValueError(f"{kind}: step {step} != {spec['expected_step']}")
    if raw.get("batch_stats"):
        raise ValueError(f"{kind}: batch_stats inesperado para MAXIM.")
    variables = {"params": raw["params"]}
    validate_tree(template_variables(spec["template_variant"]), variables, kind)
    fingerprint = hash_variables(variables)
    return {
        "variables": jax.tree_util.tree_map(jnp.asarray, variables),
        "metadata": {
            "checkpoint_path": str(checkpoint_path),
            "restored_step": step,
            "variable_fingerprint_sha256": fingerprint,
            "selection": selection,
            "split_manifest_sha256": split_hash,
        },
    }


VARIABLES = {}


def cast_floating(tree, dtype):
    return jax.tree_util.tree_map(
        lambda leaf: leaf.astype(dtype)
        if jnp.issubdtype(leaf.dtype, jnp.floating) else leaf,
        tree,
    )


def variant_variables(name):
    if name in VARIABLES:
        return VARIABLES[name]
    base = variant_base(name)
    init = VARIANTS[base]["init"]
    audit = {"init": init, "dtype": str(np.dtype(variant_dtype(name)))}
    if init in ("baseline_checkpoint", "shared_checkpoint"):
        source = "baseline" if init == "baseline_checkpoint" else "shared"
        variables = CHECKPOINTS[source]["variables"]
        validate_tree(template_variables(base), variables, name)
        audit["source_fingerprint_sha256"] = (
            CHECKPOINTS[source]["metadata"]["variable_fingerprint_sha256"]
        )
    elif init == "shared_plus_new":
        fresh = initialized_variables(base, NEW_MODULE_SEED)
        params, transplant = transplant_params(
            CHECKPOINTS["shared"]["variables"]["params"],
            fresh["params"],
            allowed_new=is_new_local_mixer,
            allowed_removed=removed_rcab_conv_at(variant_replaced_levels(base)),
        )
        variables = {"params": params}
        validate_tree(template_variables(base), variables, name)
        audit["source_fingerprint_sha256"] = (
            CHECKPOINTS["shared"]["metadata"]["variable_fingerprint_sha256"]
        )
        audit["transplant"] = {
            key: value for key, value in transplant.items()
            if not key.endswith("_paths")
        }
        audit["transplant"]["removed_paths_sample"] = transplant[
            "removed_paths"][:4]
        audit["transplant"]["new_paths_sample"] = transplant["new_paths"][:4]
        audit["new_module_seed"] = NEW_MODULE_SEED
    elif init == "random":
        variables = initialized_variables(base, NEW_MODULE_SEED)
        audit["new_module_seed"] = NEW_MODULE_SEED
    else:
        raise ValueError(init)
    variables = jax.device_put(
        cast_floating(variables, variant_dtype(name)), DEVICE
    )
    VARIABLES[name] = (variables, audit)
    return VARIABLES[name]


print("Restauración preparada; todavía no se leyó ningún checkpoint.")

In [ ]:
# 7. Inventario de parámetros: totales, por familia y activos por tarea
def parameter_inventory(name, variables):
    base = variant_base(name)
    family = variant_family(name)
    selection = VARIANTS[base].get("maxim", {}).get(
        "rcab_mixer_selection", "gather"
    )
    flat = flatten_params(variables["params"])
    total = total_bytes = semantic = executed = 0
    for path, value in flat.items():
        size = int(np.prod(value.shape))
        total += size
        total_bytes += size * np.dtype(value.dtype).itemsize
        if str(path[0]).startswith("experts_"):
            is_first = path[0] == "experts_0"
            semantic += size if is_first else 0
            if family == "shared_selective":
                executed += size if is_first else 0
            else:
                executed += size
        elif "local_mixer" in path:
            experts = int(value.shape[0])
            semantic += size // experts
            executed += size // experts if selection == "gather" else size
        else:
            semantic += size
            executed += size
    return {
        "total_parameters": total,
        "total_parameter_bytes": total_bytes,
        "semantic_active_parameters_per_task": semantic,
        "executed_parameters_per_task": executed,
        "families": parameter_families(variables["params"]),
    }

In [ ]:
# 8. Forward de inferencia y paso real de entrenamiento
def make_inference_function(name):
    model = get_model(name)
    baseline = variant_family(name) == "baseline"

    def inference(variables, batch_input, task_id):
        with jax.named_scope(f"bench_{variant_base(name)}"):
            if baseline:
                return model.apply(variables, batch_input, train=False)[-1][-1]
            predictions, _ = model.apply(
                variables, batch_input, train=False, task_id=task_id
            )
            return predictions[-1][-1]

    return jax.jit(inference)


def resize_target_to(prediction, target):
    if prediction.shape == target.shape:
        return target
    scale_h = target.shape[1] // prediction.shape[1]
    scale_w = target.shape[2] // prediction.shape[2]
    return target[:, ::scale_h, ::scale_w, :]


def compute_supervision_losses(predictions, targets, num_scales=3,
                               auxiliary_weight=1.0):
    final_prediction = predictions[-1][-1]
    final_loss = jnp.mean(jnp.abs(
        final_prediction - resize_target_to(final_prediction, targets)
    ))
    auxiliary_sum = jnp.zeros_like(final_loss)
    weight_sum = 0.0
    last_stage = len(predictions) - 1
    for stage_index, stage_predictions in enumerate(predictions):
        for scale_index, prediction in enumerate(stage_predictions):
            if (stage_index == last_stage
                    and scale_index == len(stage_predictions) - 1):
                continue
            weight = 0.5 ** (num_scales - scale_index - 1)
            auxiliary_sum += weight * jnp.mean(jnp.abs(
                prediction - resize_target_to(prediction, targets)
            ))
            weight_sum += weight
    auxiliary_loss = auxiliary_sum / max(weight_sum, 1e-8)
    return (final_loss + auxiliary_weight * auxiliary_loss, final_loss,
            auxiliary_loss)


def make_optimizer():
    return optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(learning_rate=TRAIN_LEARNING_RATE,
                    weight_decay=TRAIN_WEIGHT_DECAY),
    )


def split_trainable(params, scope):
    flat = flatten_params(params)
    if scope == "full":
        trainable, frozen = flat, {}
    elif scope == "new_only":
        trainable = {k: v for k, v in flat.items() if "local_mixer" in k}
        frozen = {k: v for k, v in flat.items() if "local_mixer" not in k}
        if not trainable:
            raise ValueError("new_only requiere módulos nuevos local_mixer.")
    else:
        raise ValueError(scope)
    count = lambda tree: int(sum(np.prod(v.shape) for v in tree.values()))
    return (
        traverse_util.unflatten_dict(trainable),
        traverse_util.unflatten_dict(frozen) if frozen else {},
        {"trainable_parameters": count(trainable),
         "frozen_parameters": count(frozen)},
    )


def merge_params(trainable, frozen):
    merged = dict(flatten_params(frozen)) if frozen else {}
    merged.update(flatten_params(trainable))
    return traverse_util.unflatten_dict(merged)


def make_train_step(name):
    model = get_model(name)
    baseline = variant_family(name) == "baseline"
    num_scales = MODEL_CONFIG["num_supervision_scales"]

    def train_step(state, frozen, batch_input, batch_target, task_id, rng):
        def loss_fn(trainable):
            params = merge_params(trainable, frozen)
            kwargs = {} if baseline else {"task_id": task_id}
            output = model.apply(
                {"params": params}, batch_input, train=True,
                rngs={"dropout": rng}, **kwargs,
            )
            predictions = output if baseline else output[0]
            loss, final_loss, auxiliary_loss = compute_supervision_losses(
                predictions, batch_target, num_scales=num_scales,
                auxiliary_weight=AUXILIARY_LOSS_WEIGHT,
            )
            return loss, (final_loss, auxiliary_loss)

        (loss, (final_loss, auxiliary_loss)), grads = jax.value_and_grad(
            loss_fn, has_aux=True
        )(state.params)
        new_state = state.apply_gradients(grads=grads)
        return new_state, {
            "loss": loss, "final_loss": final_loss,
            "auxiliary_loss": auxiliary_loss,
            "grad_norm": optax.global_norm(grads),
        }

    return jax.jit(train_step)


print("Funciones de inferencia y entrenamiento preparadas.")

In [ ]:
# 9. Pruebas sintéticas de funciones puras (no leen checkpoints)
_target = jnp.linspace(0.0, 1.0, 2 * 16 * 16 * 3,
                       dtype=jnp.float32).reshape(2, 16, 16, 3)
_predictions = [[_target[:, ::4, ::4], _target[:, ::2, ::2], _target]] * 2
np.testing.assert_allclose(
    np.asarray(compute_supervision_losses(_predictions, _target)), 0.0,
    atol=1e-7,
)
_tree = {"a": {"local_mixer": {"k": jnp.ones((2,))}, "conv": {"k": jnp.ones(3)}}}
_trainable, _frozen, _counts = split_trainable(_tree, "new_only")
assert _counts == {"trainable_parameters": 2, "frozen_parameters": 3}
assert jax.tree_util.tree_structure(merge_params(_trainable, _frozen)) == (
    jax.tree_util.tree_structure(_tree))
_hlo = (
    "HloModule m\n\nENTRY %main {\n"
    "  %a = f32[1] fusion(%p), kind=kLoop\n"
    '  %b = (f32[1]) custom-call(%a), custom_call_target="__cudnn$convForward"\n'
    "}\n"
)
assert hlo_statistics(_hlo)["kernel_launch_estimate"] == 2
assert hlo_statistics(_hlo)["conv_custom_calls"] == 1
_tmp = OUTPUT_DIR / "_atomic_json_test.json"
atomic_json(_tmp, {"x": np.float32(1.0), "p": Path("/tmp")})
assert json.loads(_tmp.read_text()) == {"x": 1.0, "p": "/tmp"}
_tmp.unlink()
print("Pruebas sintéticas OK.")

In [ ]:
# 10. Compilación, análisis de costo y cronometraje alternado
def _normalize(analysis):
    if analysis is None:
        return {"available": False, "error": "backend returned None"}
    if isinstance(analysis, (list, tuple)):
        analysis = analysis[0] if len(analysis) == 1 else {
            f"entry_{index}": item for index, item in enumerate(analysis)
        }
    if hasattr(analysis, "_asdict"):
        analysis = analysis._asdict()
    if not isinstance(analysis, Mapping):
        fields = [
            "generated_code_size_in_bytes", "argument_size_in_bytes",
            "output_size_in_bytes", "alias_size_in_bytes",
            "temp_size_in_bytes",
        ]
        analysis = {field: getattr(analysis, field)
                    for field in fields if hasattr(analysis, field)}
    return {"available": True, "values": json_safe(dict(analysis))}


def _safe(call):
    try:
        return _normalize(call())
    except Exception as error:
        return {"available": False,
                "error": f"{type(error).__name__}: {error}"}


def save_hlo_text(text, key):
    if not SAVE_HLO or text is None:
        return None
    HLO_DIR.mkdir(parents=True, exist_ok=True)
    destination = HLO_DIR / (key.replace("/", "__").replace(":", "_")
                             + ".txt.gz")
    with gzip.open(destination, "wt", encoding="utf-8") as handle:
        handle.write(text)
    return {"path": str(destination),
            "sha256": hashlib.sha256(text.encode("utf-8")).hexdigest()}


def compile_entry(jitted, args, key):
    started = time.perf_counter()
    lowered = jitted.lower(*args)
    lowered_cost = _safe(lowered.cost_analysis)
    compiled = lowered.compile()
    seconds = time.perf_counter() - started
    try:
        text = compiled.as_text()
    except Exception:
        text = None
    sample = block_tree(compiled(*args))
    return {
        "compiled": compiled,
        "sample": sample,
        "compilation_seconds": seconds,
        "lowered_cost_analysis": lowered_cost,
        "compiled_cost_analysis": _safe(compiled.cost_analysis),
        "memory_analysis": _safe(compiled.memory_analysis),
        "hlo_statistics": hlo_statistics(text) if text else None,
        "hlo": save_hlo_text(text, key),
    }


def run_interleaved(entries):
    """entries: nombre -> {"compiled", "arg_sets"}. Tiempos en ms."""
    names = list(entries)
    for name in names:
        compiled, arg_sets = entries[name]["compiled"], entries[name]["arg_sets"]
        for index in range(WARMUP_EXECUTIONS):
            block_tree(compiled(*arg_sets[index % len(arg_sets)]))
    samples = {name: [] for name in names}
    blocks = {name: [] for name in names}
    orders = []
    for block in range(TIMING_BLOCKS):
        shift = block % len(names)
        order = names[shift:] + names[:shift]
        if block % 2 == 1:
            order = order[::-1]
        orders.append(order)
        for position, name in enumerate(order):
            compiled, arg_sets = (entries[name]["compiled"],
                                  entries[name]["arg_sets"])
            block_tree(compiled(*arg_sets[0]))  # recalentar tras cambiar
            values = []
            for index in range(TIMED_EXECUTIONS):
                args = arg_sets[index % len(arg_sets)]
                started = time.perf_counter_ns()
                block_tree(compiled(*args))
                values.append((time.perf_counter_ns() - started) / 1e6)
            samples[name].extend(values)
            blocks[name].append({
                "block": block + 1, "position": position,
                "median_ms": float(np.median(values)),
                "p95_ms": float(np.percentile(values, 95)),
                "mean_ms": float(np.mean(values)),
            })
            print(f"  bloque {block + 1} {name}: "
                  f"{np.median(values):.3f} ms")
    results = {}
    for name in names:
        values = samples[name]
        medians = [item["median_ms"] for item in blocks[name]]
        median = float(np.median(values))
        results[name] = {
            "median_ms": median,
            "p95_ms": float(np.percentile(values, 95)),
            "mean_ms": float(np.mean(values)),
            "block_median_spread_pct": float(
                100 * (max(medians) - min(medians)) / median),
            "blocks": blocks[name],
            "executions": len(values),
        }
    return results, orders

In [ ]:
# 11. Grupo de inferencia: todas las variantes de una resolución juntas
def conv_metric(stats):
    if not stats:
        return None
    return stats["conv_custom_calls"] or stats["convolution_ops"]


def max_task_difference(first, second):
    return float(max(
        np.max(np.abs(a.astype(np.float32) - b.astype(np.float32)))
        for a, b in zip(first, second)
    ))


def run_inference_group(shape):
    height, width = shape
    group = stable_key("inference", f"{height}x{width}")
    if group_complete(group):
        print("Grupo completo, se omite:", group)
        return
    print("\n=== Grupo", group, "===")
    REPORT["groups"][group] = {"status": "running"}
    persist_report()

    rng = np.random.default_rng(SEED)
    batch = jax.device_put(
        rng.random((1, height, width, 3), dtype=np.float32), DEVICE)
    task_ids = [jax.device_put(jnp.full((1,), index, dtype=jnp.int32), DEVICE)
                for index in range(len(TASKS))]
    entries, payloads, outputs = {}, {}, {}
    for name in INFERENCE_VARIANTS:
        print("Compilando", name)
        variables, _ = variant_variables(name)
        batch_input = batch.astype(variant_dtype(name))
        uses_task = variant_family(name) != "baseline"
        arg_sets = ([(variables, batch_input, task) for task in task_ids]
                    if uses_task else [(variables, batch_input, task_ids[0])])
        payload = compile_entry(
            make_inference_function(name), arg_sets[0],
            stable_key(group, name))
        payload.pop("sample")
        outputs[name] = [
            np.asarray(block_tree(payload["compiled"](*args)))
            for args in arg_sets
        ]
        entries[name] = {"compiled": payload["compiled"], "arg_sets": arg_sets}
        payloads[name] = payload

    passed = []
    for name, values in outputs.items():
        passed.append(record_check(
            group, f"{name}: salida finita",
            all(np.isfinite(value.astype(np.float32)).all() for value in values),
            {"shape": list(values[0].shape)}))
    pairs = [("S0_sel", "S0", "selección efectiva de la cabeza"),
             ("S2_d32", "S2_d32_todos", "selección efectiva del banco")]
    for candidate, reference, label in pairs:
        if candidate in outputs and reference in outputs:
            difference = max_task_difference(outputs[candidate],
                                             outputs[reference])
            passed.append(record_check(
                group, f"{candidate} ≡ {reference} ({label})",
                difference <= EQUIVALENCE_TOLERANCE,
                {"max_abs_diff": difference,
                 "tolerance": EQUIVALENCE_TOLERANCE}))
    for name in outputs:
        if name.endswith("@bf16") and variant_base(name) in outputs:
            reference = outputs[variant_base(name)]
            difference = max_task_difference(outputs[name], reference)
            mean_difference = float(np.mean([
                np.mean(np.abs(a.astype(np.float32) - b))
                for a, b in zip(outputs[name], reference)]))
            record_check(group, f"{name}: desvío frente a float32", True,
                         {"max_abs_diff": difference,
                          "mean_abs_diff": mean_difference,
                          "nota": "desvío numérico, no calidad"})
    convs = {name: conv_metric(payload["hlo_statistics"])
             for name, payload in payloads.items()}
    structural = [
        ("S0", "S0_sel", lambda a, b: a - b == 12,
         "S0 tiene 12 convoluciones más: 4 cabezas no seleccionadas × 3"),
        ("S2_d32", "S1_d32", lambda a, b: a == b,
         "S2 ejecuta tantas convoluciones como S1"),
        ("S2_d32_todos", "S2_d32", lambda a, b: a > b,
         "calcular todos los expertos agrega convoluciones"),
    ]
    for first, second, rule, label in structural:
        if convs.get(first) is not None and convs.get(second) is not None:
            passed.append(record_check(
                group, label, rule(convs[first], convs[second]),
                {first: convs[first], second: convs[second]}))
    persist_report()
    if STRICT_CHECKS and not all(passed):
        REPORT["groups"][group]["status"] = "checks_failed"
        persist_report()
        raise AssertionError(f"Fallaron verificaciones en {group}.")

    timings, orders = run_interleaved(entries)
    for name in INFERENCE_VARIANTS:
        payload = payloads[name]
        REPORT["measurements"][stable_key(group, name)] = {
            "status": "complete",
            "group": group,
            "variant": name,
            "dtype": str(np.dtype(variant_dtype(name))),
            "shape": [height, width],
            "batch_size": 1,
            "task_mode": ("ciclo de las cinco tareas"
                          if variant_family(name) != "baseline"
                          else "sin tarea"),
            **{key: payload[key] for key in (
                "compilation_seconds", "lowered_cost_analysis",
                "compiled_cost_analysis", "memory_analysis",
                "hlo_statistics", "hlo")},
            "timing": timings[name],
        }
    REPORT["groups"][group] = {
        "status": "complete", "orders": orders,
        "device_memory_stats_after": device_memory_stats(),
    }
    persist_report()
    del entries, payloads, outputs
    gc.collect()

In [ ]:
# 12. Grupo de entrenamiento: pasos reales con batch mixto de tareas
class BenchmarkState(train_state.TrainState):
    pass


def run_training_group():
    height, width = TRAIN_SHAPE
    group = stable_key("training", f"{height}x{width}_batch{TRAIN_BATCH_SIZE}")
    if group_complete(group):
        print("Grupo completo, se omite:", group)
        return
    print("\n=== Grupo", group, "===")
    REPORT["groups"][group] = {"status": "running"}
    persist_report()

    rng_np = np.random.default_rng(SEED + 100)
    shape = (TRAIN_BATCH_SIZE, height, width, 3)
    batch_input = jax.device_put(rng_np.random(shape, dtype=np.float32), DEVICE)
    batch_target = jax.device_put(rng_np.random(shape, dtype=np.float32), DEVICE)
    task_ids = jax.device_put(np.asarray(TRAIN_TASK_IDS, dtype=np.int32), DEVICE)
    rng = jax.random.PRNGKey(SEED + 200)

    entries, payloads, info = {}, {}, {}
    passed = []
    for name, scope in TRAINING_RUNS:
        label = f"{name}:{scope}"
        print("Compilando", label)
        variables, _ = variant_variables(name)
        trainable, frozen, counts = split_trainable(variables["params"], scope)
        state = BenchmarkState.create(
            apply_fn=get_model(name).apply, params=trainable,
            tx=make_optimizer())
        args = (state, frozen, batch_input, batch_target, task_ids, rng)
        payload = compile_entry(make_train_step(name), args,
                                stable_key(group, label))
        new_state, metrics = payload.pop("sample")
        loss = float(metrics["loss"])
        passed.append(record_check(
            group, f"{label}: pérdida finita y un paso de optimizador",
            np.isfinite(loss) and int(new_state.step) == int(state.step) + 1,
            {"loss": loss, "grad_norm": float(metrics["grad_norm"])}))
        info[label] = {"variant": name, "scope": scope, "sample_loss": loss,
                       **counts}
        entries[label] = {"compiled": payload["compiled"], "arg_sets": [args]}
        payloads[label] = payload
        del new_state, metrics

    if "B0:full" in info and "B0_remat:full" in info:
        difference = abs(info["B0:full"]["sample_loss"]
                         - info["B0_remat:full"]["sample_loss"])
        passed.append(record_check(
            group, "rematerialización conserva la pérdida", difference <= 1e-5,
            {"abs_diff": difference}))
    persist_report()
    if STRICT_CHECKS and not all(passed):
        REPORT["groups"][group]["status"] = "checks_failed"
        persist_report()
        raise AssertionError(f"Fallaron verificaciones en {group}.")

    timings, orders = run_interleaved(entries)
    for label, payload in payloads.items():
        REPORT["measurements"][stable_key(group, label)] = {
            "status": "complete",
            "group": group,
            **info[label],
            "shape": [height, width],
            "batch_size": TRAIN_BATCH_SIZE,
            "task_ids": TRAIN_TASK_IDS,
            "optimizer": "clip_by_global_norm(1.0) + AdamW",
            **{key: payload[key] for key in (
                "compilation_seconds", "lowered_cost_analysis",
                "compiled_cost_analysis", "memory_analysis",
                "hlo_statistics", "hlo")},
            "timing": timings[label],
        }
    REPORT["groups"][group] = {
        "status": "complete", "orders": orders,
        "device_memory_stats_after": device_memory_stats(),
    }
    persist_report()
    del entries, payloads
    gc.collect()

In [ ]:
# 13. Orquestación
try:
    names = list(dict.fromkeys(
        (INFERENCE_VARIANTS if RUN_INFERENCE else [])
        + ([run[0] for run in TRAINING_RUNS] if RUN_TRAINING else [])
    ))
    inits = {VARIANTS[variant_base(name)]["init"] for name in names}
    needed = set()
    if "baseline_checkpoint" in inits:
        needed.add("baseline")
    if inits & {"shared_checkpoint", "shared_plus_new"}:
        needed.add("shared")
    for kind in sorted(needed):
        print("Restaurando checkpoint", kind)
        CHECKPOINTS[kind] = load_checkpoint_strict(kind)
        REPORT["checkpoints"][kind] = CHECKPOINTS[kind]["metadata"]
    split_hashes = {payload["split_manifest_sha256"]
                    for payload in REPORT["checkpoints"].values()}
    if len(split_hashes) > 1:
        raise ValueError("Baseline y shared no usan el mismo split manifest.")
    for name in names:
        variables, audit = variant_variables(name)
        REPORT["variants"][name] = {
            "audit": audit,
            "parameters": parameter_inventory(name, variables),
        }
        print(name, REPORT["variants"][name]["parameters"]["total_parameters"])
    persist_report()

    if RUN_INFERENCE:
        for inference_shape in INFERENCE_SHAPES:
            run_inference_group(inference_shape)
    if RUN_TRAINING:
        run_training_group()

    all_groups = [stable_key("inference", f"{h}x{w}") for h, w in INFERENCE_SHAPES]
    all_groups.append(stable_key(
        "training", f"{TRAIN_SHAPE[0]}x{TRAIN_SHAPE[1]}_batch{TRAIN_BATCH_SIZE}"))
    done = all(REPORT["groups"].get(group, {}).get("status") == "complete"
               for group in all_groups)
    REPORT["status"] = "complete" if done else "partial"
    REPORT["executions"][-1]["status"] = "complete"
    persist_report()
    print("Estado:", REPORT["status"])
except Exception as error:
    REPORT["executions"][-1].update({
        "status": "error",
        "error": f"{type(error).__name__}: {error}",
        "traceback": traceback.format_exc(),
    })
    persist_report()
    raise

In [ ]:
# 14. Reporte legible derivado del JSON, sin decisiones
def analysis_value(analysis, key):
    if not analysis or not analysis.get("available"):
        return None
    value = analysis.get("values", {}).get(key)
    return float(value) if isinstance(value, (int, float)) else None


def fmt(value, scale=1.0, digits=2):
    if value is None:
        return "—"
    return f"{value / scale:,.{digits}f}".replace(",", " ")


def pct(value, reference):
    if value is None or not reference:
        return "—"
    return f"{100 * (value - reference) / reference:+.1f} %"


def build_report():
    lines = [
        "# Benchmark v2: compresión de MAXIM con expertos",
        "",
        f"- Estado: **{REPORT['status']}**",
        f"- Protocolo: `{PROTOCOL_SHA256}`",
        f"- Commit: `{REPORT['environment']['repo_commit']}`",
        f"- GPU: {REPORT['environment']['nvidia_smi']}",
        "- Sólo costo. Los módulos nuevos de S1/S2 tienen pesos aleatorios.",
        "",
        "## Parámetros",
        "",
        "| Variante | Totales | Usados por tarea | Ejecutados por tarea "
        "| Δ totales vs S0_sel | Δ totales vs B0 |",
        "|---|---:|---:|---:|---:|---:|",
    ]
    variants = REPORT["variants"]
    total = lambda name: variants.get(name, {}).get(
        "parameters", {}).get("total_parameters")
    for name, payload in variants.items():
        parameters = payload["parameters"]
        lines.append(
            f"| {name} | {parameters['total_parameters']:,} | "
            f"{parameters['semantic_active_parameters_per_task']:,} | "
            f"{parameters['executed_parameters_per_task']:,} | "
            f"{pct(parameters['total_parameters'], total('S0_sel'))} | "
            f"{pct(parameters['total_parameters'], total('B0'))} |"
        )

    for group, status in REPORT["groups"].items():
        rows = [m for m in REPORT["measurements"].values()
                if m.get("group") == group]
        if not rows:
            continue
        lines += ["", f"## {group} ({status.get('status')})", ""]
        is_training = group.startswith("training")
        label_of = (lambda m: f"{m['variant']}:{m['scope']}") if is_training \
            else (lambda m: m["variant"])
        reference = {label_of(m): m["timing"]["median_ms"] for m in rows}
        base_label = "B0:full" if is_training else "B0"
        lines += [
            "| Variante | Mediana ms | p95 ms | Dispersión entre bloques "
            "| Δ vs B0 | Δ vs S0_sel | GFLOP antes de optimizar "
            "| GFLOP compilado | GB accedidos | Kernels est. | Temp. MiB |",
            "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|",
        ]
        for m in rows:
            label = label_of(m)
            timing = m["timing"]
            sel_label = "S0_sel:full" if is_training else "S0_sel"
            stats = m.get("hlo_statistics") or {}
            lines.append(
                f"| {label} | {timing['median_ms']:.2f} | "
                f"{timing['p95_ms']:.2f} | "
                f"{timing['block_median_spread_pct']:.1f} % | "
                f"{pct(timing['median_ms'], reference.get(base_label))} | "
                f"{pct(timing['median_ms'], reference.get(sel_label))} | "
                f"{fmt(analysis_value(m['lowered_cost_analysis'], 'flops'), 1e9)} | "
                f"{fmt(analysis_value(m['compiled_cost_analysis'], 'flops'), 1e9)} | "
                f"{fmt(analysis_value(m['compiled_cost_analysis'], 'bytes accessed'), 1e9)} | "
                f"{stats.get('kernel_launch_estimate', '—')} | "
                f"{fmt(analysis_value(m['memory_analysis'], 'temp_size_in_bytes'), 2**20, 0)} |"
            )
        if is_training:
            lines += ["", "| Corrida | Entrenables | Congelados |",
                      "|---|---:|---:|"]
            for m in rows:
                lines.append(f"| {label_of(m)} | {m['trainable_parameters']:,} "
                             f"| {m['frozen_parameters']:,} |")

    lines += ["", "## Verificaciones", ""]
    for group, checks in REPORT["checks"].items():
        for name, check in checks.items():
            mark = "OK" if check["passed"] else "FALLA"
            lines.append(f"- {mark} · {group} · {name} · "
                         f"`{json.dumps(check['detail'], ensure_ascii=False)}`")
    lines += ["", "## Limitaciones", ""]
    lines += [f"- {item}" for item in REPORT["limitations"]]
    lines += [
        "",
        "## Próximo paso",
        "",
        "Entregar `benchmark_v2_results.json`, este reporte y la carpeta "
        "`hlo/`. La elección de variantes para recuperar calidad se hace "
        "después de revisarlos.",
    ]
    return "\n".join(lines) + "\n"


REPORT_PATH.write_text(build_report(), encoding="utf-8")
print(REPORT_PATH.read_text(encoding="utf-8"))

## Ejecución y entrega

1. Antes de abrir el notebook, subir al repositorio los archivos
   `maxim/models/maxim.py`, `maxim/models/compression.py` y
   `maxim/models/compat.py`. La celda 1 verifica sus hashes y se detiene si
   no coinciden con la versión revisada.
2. Abrir el notebook en Colab con GPU **L4** y ejecutar todas las celdas en
   orden. Si la celda 1 instala paquetes después de que JAX ya estaba
   importado, reiniciar el runtime y volver a ejecutar todo.
3. Si el runtime se corta, reconectar y ejecutar todo de nuevo con la misma
   GPU. Los grupos completos se omiten y los demás se repiten enteros.
4. Para dividir la corrida en dos sesiones, ejecutar primero con
   `RUN_TRAINING = False` y luego con `RUN_INFERENCE = False`. Estos dos
   indicadores no cambian el protocolo.
5. Si una verificación falla, el grupo se detiene antes de cronometrar y el
   motivo queda en `checks` del JSON.

Entregar desde `ckpts/benchmark_maxim_v2_compresion/`:

- `benchmark_v2_results.json`;
- `diagnostico_v2_compresion.md`;
- la carpeta `hlo/`, comprimida;
- la salida de consola si hubo errores.

No interpretar estos números como calidad. Los FLOPs "antes de optimizar" son
los comparables entre variantes; la latencia es la que importa para tiempo; los
parámetros totales y por tarea son los que importan para tamaño.